In [3]:
# Day 2 — Data Cleaning & Validation
## Bluestock Fintech | Mutual Fund Analytics Platform

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path().resolve().parent
print("Libraries loaded!")

Libraries loaded!


In [5]:
df = pd.read_csv(BASE / "data/raw/02_nav_history.csv")
print("Raw shape:", df.shape)

df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')
df = df.sort_values(['amfi_code','date']).reset_index(drop=True)
df['nav'] = df.groupby('amfi_code')['nav'].ffill()

before = len(df)
df = df.drop_duplicates(subset=['amfi_code','date'])
print(f"Duplicates removed: {before - len(df)}")

df = df[df['nav'] > 0]
df['daily_return_pct'] = df.groupby('amfi_code')['nav'].pct_change() * 100

df.to_csv(BASE / "data/processed/clean_nav.csv", index=False)
print(f"Clean NAV shape: {df.shape}")
df.head()

Raw shape: (46000, 3)
Duplicates removed: 0
Clean NAV shape: (46000, 4)


,amfi_code,date,nav,daily_return_pct
0,100016,2022-01-03,520.4608,NaN
1,100016,2022-01-04,515.0971,-1.030568
2,100016,2022-01-05,521.7239,1.286515
3,100016,2022-01-06,515.7880,-1.137747
4,100016,2022-01-07,515.1639,-0.120999


In [6]:
tx = pd.read_csv(BASE / "data/raw/08_investor_transactions.csv")
print("Raw shape:", tx.shape)

tx['transaction_type'] = tx['transaction_type'].str.strip().str.title()
type_map = {'Sip':'SIP','Lumpsum':'Lumpsum','Lump Sum':'Lumpsum',
            'Redemption':'Redemption','Redeem':'Redemption'}
tx['transaction_type'] = tx['transaction_type'].replace(type_map)
tx = tx[tx['amount_inr'] > 0]
tx['date'] = pd.to_datetime(tx['transaction_date'], format='%Y-%m-%d')

print("Transaction types:")
print(tx['transaction_type'].value_counts())
print("\nKYC Status:")
print(tx['kyc_status'].value_counts())

tx.to_csv(BASE / "data/processed/clean_transactions.csv", index=False)
print(f"\nClean transactions shape: {tx.shape}")

Raw shape: (32778, 13)
Transaction types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC Status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Clean transactions shape: (32778, 14)


In [7]:
perf = pd.read_csv(BASE / "data/raw/07_scheme_performance.csv")
print("Raw shape:", perf.shape)

return_cols = [c for c in perf.columns if 'return' in c.lower()]
for col in return_cols:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')

exp_col = [c for c in perf.columns if 'expense' in c.lower()][0]
out_of_range = perf[(perf[exp_col] < 0.1) | (perf[exp_col] > 2.5)]
print(f"Expense ratios out of range: {len(out_of_range)}")

num_cols = perf.select_dtypes(include=np.number).columns
perf[num_cols] = perf[num_cols].fillna(perf[num_cols].median())

perf.to_csv(BASE / "data/processed/clean_performance.csv", index=False)
print(f"Clean performance shape: {perf.shape}")
perf.head()

Raw shape: (40, 19)
Expense ratios out of range: 0
Clean performance shape: (40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


## Data Quality Summary
- NAV History: 46,000 rows, 0 duplicates, forward-filled holiday gaps
- Transactions: 32,778 rows, standardised transaction types, removed zero amounts
- Performance: 40 rows, all expense ratios within valid range (0.1% - 2.5%)
- All 3 files saved to data/processed/